In [ ]:
!pip install requests


In [ ]:
pip install groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.8/130.8 kB 4.9 MB/s eta 0:00:00


In [ ]:
import re
import os
from bs4 import BeautifulSoup
import requests
from groq import Groq

# Setup Groq client
client = Groq(api_key="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")

def get_top_yahoo_finance_news():
    headers = {
        'accept': '*/*',
        'accept-encoding': 'gzip, deflate, br',
        'accept-language': 'en-US,en;q=0.9',
        'referer': 'https://www.google.com',
        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
    }

    url = 'https://news.search.yahoo.com/search?p=site:finance.yahoo.com'
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    card = soup.find('div', 'NewsArticle')

    if card:
        headline = card.find('h4', 's-title').text
        description = card.find('p', 's-desc').text.strip()
        raw_link = card.find('a').get('href')
        unquoted_link = requests.utils.unquote(raw_link)
        pattern = re.compile(r'RU=(.+)\/RK')
        match = re.search(pattern, unquoted_link)
        clean_link = match.group(1) if match else raw_link
        return {
            'headline': headline,
            'description': description,
            'link': clean_link
        }
    return None

def generate_linkedin_post_with_groq_sdk(news):
    prompt = f"""
Write a LinkedIn post in my style — casual, clear, and scroll-stopping.
It should start with a strong hook(it should be seperate from the content), explain the idea simply but insightfully, and end with a relatable line that invites interaction (no begging).
Avoid emojis, sound like a smart investor and add relevant hashtags that make smart people want to connect.
Also should sound like a human not AI.

Here is the article:

Headline: {news['headline']}
Description: {news['description']}
Link: {news['link']}
"""

    chat_completion = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ]
    )

    return chat_completion.choices[0].message.content


In [ ]:
news = get_top_yahoo_finance_news()
if news:
    linkedin_post = generate_linkedin_post_with_groq_sdk(news)
    print("📢 LinkedIn Post:\n")
    print(linkedin_post)
else:
    print("No news found.")


📢 LinkedIn Post:

Here's a LinkedIn post in your style:

**Hook:** "Tax reform just got a whole lot more interesting..."

When it comes to the proposed tax bill, one thing is clear: the stakes are high, and the clock is ticking. President Trump's latest push to speed up the process has many wondering what this means for the future of our economy.

At the heart of the matter is the tentative deal reached on the State and Local Tax (SALT) deduction, a crucial component of the tax overhaul. While the details are still emerging, one thing is certain - the next few weeks will be pivotal in shaping the trajectory of our economic landscape.

As investors, we're constantly weighing risk and opportunity. But in this moment, it's not just about the numbers - it's about the future we're building for ourselves and for generations to come.

So, what do you think the real impact of this tax reform will be? Are you optimistic about the potential benefits, or do you have concerns about the unintended 